# Emotion Detection from Text (NLP)
This notebook focuses on classifying text comments into various emotions using LSTM.

In [ ]:
import numpy as np
import pandas as pd
import pickle
import nltk
import re
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import seaborn as sns
import matplotlib.pyplot as plt
from wordcloud import WordCloud

nltk.download('stopwords')

## Load and Explore Data

In [ ]:
# Ensure you have 'train.txt' in the same directory
try:
    train_data = pd.read_csv("train.txt", header=None, sep=";", names=["Comment", "Emotion"], encoding="utf-8")
    print("Data loaded successfully")
    print(train_data.head())
except FileNotFoundError:
    print("train.txt not found. Please provide the dataset.")

In [ ]:
sns.countplot(x=train_data['Emotion'])
plt.title("Emotion Distribution")
plt.show()

## Data Preprocessing

In [ ]:
lb = LabelEncoder()
train_data['Emotion_Label'] = lb.fit_transform(train_data['Emotion'])

stop_words = set(stopwords.words('english'))

def text_cleaning(df, column, vocab_size, max_len):
    stemmer = PorterStemmer()
    corpus = []
    for text in df[column]:
        text = re.sub("[^a-zA-Z]", " ", text)
        text = text.lower()
        text = text.split()
        text = [stemmer.stem(word) for word in text if word not in stop_words]
        text = " ".join(text)
        corpus.append(text)
    
    one_hot_word = [one_hot(input_text=word, n=vocab_size) for word in corpus]
    pad = pad_sequences(sequences=one_hot_word, maxlen=max_len, padding='pre')
    return pad

vocab_size = 11000
max_len = 300
X = text_cleaning(train_data, "Comment", vocab_size, max_len)
Y = to_categorical(train_data["Emotion_Label"])

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

## Model Building

In [ ]:
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=150, input_length=max_len),
    Dropout(0.2),
    LSTM(128),
    Dropout(0.2),
    Dense(64, activation='sigmoid'),
    Dropout(0.2),
    Dense(6, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

callback = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=10, batch_size=64, validation_data=(X_test, y_test), callbacks=[callback])

In [ ]:
model.save('emotion_model.h5')
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(lb, f)